# Part 1: Sequence Modelling

__Before starting, we recommend you enable GPU acceleration if you're running on Colab.__

In [ ]:
# Execute this code block to install dependencies
try:
    import torchbearer
except ImportError:
    !pip install torchbearer

In [ ]:
#imports
import collections
from torchvision.datasets.utils import download_url
import torch
import io


#RNN imports
from torch import nn
from torch.nn import functional as F
import random
import sys
from torch.utils.data import Dataset, DataLoader
from torch import optim
from torch import nn

# training imports
import torchbearer
from torchbearer import Trial
from torchbearer.callbacks.decorators import on_end_epoch


## Markov chains

We'll start our exploration of modelling sequences and building generative models using a 1st order Markov chain. The Markov chain is a stochastic model describing a sequence of possible events in which the probability of each event depends only on the state attained in the previous event. In our case we're going to learn a model over a set of characters from an English language text. The events, or states, in our model are the set of possible characters, and we'll learn the probability of moving from one character to the next.

Let's start by loading the data from the web:

In [ ]:
# Read the data
download_url('https://s3.amazonaws.com/text-datasets/nietzsche.txt', '.', 'nietzsche.txt', None)
text = io.open('./nietzsche.txt', encoding='utf-8').read().lower()
print('Corpus length:', len(text))

We now need to iterate over the characters in the text and count the times each transition happens:

In [ ]:
transition_counts = collections.defaultdict(dict)
for i in range(0,len(text)-1):
    curr_ch, next_ch = text[i], text[i+1]
    transition_counts[curr_ch][next_ch] = transition_counts[curr_ch].get(next_ch, 0) + 1

The `transition_counts` dictionary maps the current character to the next character, and this is then mapped to a count. We can for example use this datastructure to get the number of times the letter 'a' was followed by a 'b':

In [ ]:
print("Number of transitions from 'a' to 'b': " + str(transition_counts['a']['b']))

Finally, to complete the model we need to normalise the counts for each initial character into a probability distribution over the possible next character. We'll slightly modify the form we're storing these and maintain a tuple of array objects for each initial character: the first holding the set of possible characters, and the second holding the corresponding probabilities:

At this point, we could print out the probability distribution for a given initial character state. For example, to print the distribution for 'a':

In [ ]:
transition_probabilities = dict()

for curr_ch, next_characters in transition_counts.items():

    values = list(next_characters.keys())
    probabilities = list(next_characters.values())
    norm_prob = [prob/sum(probabilities) for prob in probabilities]

    transition_probabilities[curr_ch] = (values, norm_prob)

In [ ]:
print('NEXT CH', '->', 'PROBABILITY')
for a,b in zip(transition_probabilities['a'][0], transition_probabilities['a'][1]):
    print(a, '      ->', b)

It looks like the most probable letter to follow an 'a' is 'n'. 

__What is the most likely letter to follow the letter 'j'? Write your answer in the block below:__

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

We mentioned earlier that the Markov model is generative. This means that we can draw samples from the distributions and iteratively move between states. 

Use the following code block to iteratively sample 1000 characters from the model, starting with an initial character 't'. You can use the `torch.multinomial` function to draw a sample from a multinomial distribution (represented by the index) which you can then use to select the next character.

In [ ]:
current = 't'
for i in range(0, 1000):
    print(current, end='')
    # sample the next character based on `current` and store the result in `current`
    # YOUR CODE HERE
    raise NotImplementedError()

## Questions and exercises

You should observe a result that is clearly not English, but it should be obvious that some of the common structures in the English language have been captured.

__Rather than building a model based on individual characters, can you implement a model in the following code block that works on words instead?__

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

__Can we restrict the model from outputting certain words?__

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## RNN-based sequence modelling

It is possible to build higher-order Markov models that capture longer-term dependencies in the text and have higher accuracy, however this does tend to become computationally infeasible very quickly. Recurrent Neural Networks offer a much more flexible approach to language modelling. 

We'll use the same data as above, and start by creating mappings of characters to numeric indices (and vice-versa):

In [ ]:
chars = sorted(list(set(text)))
print('total chars:', len(chars))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

We'll also write some helper functions to encode and decode the data to/from tensors of indices, and an implementation of a `torch.Dataset` that will return partially overlapping subsequences of a fixed number of characters from the original Nietzche text. Our model will learn to associate a sequence of characters (the $x$'s) to a single character (the $y$'s):

In [ ]:
MAX_LEN = 40
STEP = 3

def encode(inp):
    # encode the characters in a tensor
    x = torch.zeros(MAX_LEN, dtype=torch.long)
    for t, char in enumerate(inp):
        x[t] = char_indices[char]

    return x

def decode(ten):
    s = ''
    for v in ten:
        s += indices_char[v] 
    return s


class MyDataset(Dataset):
    # cut the text in semi-redundant sequences of maxlen characters
    def __len__(self):
        # the number of possible subsequences of length MAX_LEN
        return (len(text) - MAX_LEN) // STEP

    def __getitem__(self, i):
        # inp is a list of MAX_LEN characters
        inp = text[i*STEP: i*STEP + MAX_LEN]
        # out is the first character after our input seq
        out = text[i*STEP + MAX_LEN]

        x = encode(inp)
        y = char_indices[out]

        return x, y

We can now define the model. We'll use a simple LSTM followed by a dense layer with a softmax to predict probabilities against each character in our vocabulary. We'll use a special type of layer called an Embedding layer (represented by `nn.Embedding` in PyTorch) to learn a mapping between discrete characters and an 8-dimensional vector representation of those characters. You'll learn more about Embeddings in the next part of the lab.

In [ ]:
class CharPredictor(nn.Module):
    def __init__(self):
        super(CharPredictor, self).__init__()
        self.emb = nn.Embedding(len(chars), 8)
        self.lstm = nn.LSTM(8, 128, batch_first=True)
        self.lin = nn.Linear(128, len(chars))

    def forward(self, x):
        x = self.emb(x)
        lstm_out, _ = self.lstm(x)
        out = self.lin(lstm_out[:,-1]) #we want the final timestep output
        return out

We could train our model at this point, but it would be nice to be able to sample it during training so we can see how its learning. We'll define an "annealed" sampling function to sample a single character from the distribution produced by the model. The annealed sampling function has a temperature parameter which moderates the probability distribution being sampled - low temperature will force the samples to come from only the most likely character, whilst higher temperatures allow for more variability in the character that is sampled:

In [ ]:
def sample(logits, temperature=1.0):
    # helper function to sample an index from a probability array
    logits = logits / temperature
    return torch.multinomial(F.softmax(logits, dim=0), 1)

Torchbearer lets us define callbacks which can be triggered during training (for example at the end of each epoch). Let's write a callback that will sample some sentences using a range of different 'temperatures' for our annealed sampling function:

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

def create_samples(epoch=None, model=None):
    with torch.no_grad():
        print('\n', f'----- GENERATING TEXT AT EPOCH {epoch} -----', flush=True)

        start_index = random.randint(0, len(text) - MAX_LEN - 1)
        for diversity in [0.2, 0.5, 1.0, 1.2]:

            start_sentence = text[start_index:start_index+MAX_LEN]
            print('\n', '\n','-- WITH SENTENCE SEED: "' + start_sentence + '"', flush=True)
            print('-- USING DIVERSITY:', diversity, '\n', '\n',  flush=True)

            # unsqueeze to create Batch dimension = 1
            inputs = encode(start_sentence).unsqueeze(0).to(device)
            for i in range(400):
                tag_scores = model(inputs)
                # 0 because there is a single seq as input, Batch = 1
                c = sample(tag_scores[0], diversity)  
                print(indices_char[c.item()], flush=True, end='')

                #adding new generated character to input
                inputs = torch.cat([inputs, c.unsqueeze(0).to(device)], dim=1)
                # remove first character
                inputs = inputs[:, 1:]

@on_end_epoch
def create_samples_torchbearer(state=None):
  epoch = -1
  if state is not None:
    epoch = state[torchbearer.EPOCH]
  create_samples(epoch=epoch, model=model)

Now, all the pieces are in place. __Use the following block to:__

- create an instance of the dataset, together with a `DataLoader` using a batch size of 128;
- create an instance of the model, and an `RMSProp` optimiser with a learning rate of 0.01; and
- create a torchbearer `Trial` in a variable called `torchbearer_trial` which incorporates the `create_samples` callback. Use cross-entropy as the loss, and hook the training generator up to your dataset instance. Make sure you move your `Trial` object to the GPU if one is available.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

Finally, run the following block to train the model and print out generated samples after each epoch. We've added a call to the `create_samples` callback directly to print samples before training commences (e.g. with random weights). Be aware this will take some time to run...

In [ ]:
create_samples_torchbearer.on_end_epoch(None)
torchbearer_trial.run(epochs=10)

## Questions and exercises

Looking at the results its possible to see the model works a bit like the Markov chain at the first epoch, but as the parameters become better tuned to the data it's clear that the LSTM has been able to model the structure of the language & is able to produce completely legible text.

__Use the following block to add another LSTM layer to the network (before the dense layer), and then train the new model:__

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

 __How does the additional layer affect performance of the model? Provide your answer in the block below:__

In [ ]:
#YOUR ANSWER HERE

__In the previous example, every sentence in the batch has the same length of MAX_LEN. However, in most cases we work with sentences that do not have the same dimension.__
__If we generate sentences of random length up to MAX_LEN, what is the most efficient way of feeding the batched input to the LSTM (see pytorch documentation)?__


In [ ]:
#YOUR ANSWER HERE

__Would making this LSTM bidirectional be appropriate for next-character generation? Explain why or why not. In what kind of sequence task might access to both previous and future context be useful?__

In [ ]:
# YOUR ANSWER HERE

__What is the one big advantage of the Markov chain model? What is the one big advantage of the LSTM model?__

In [ ]:
#YOUR ANSWER HERE